# BIKV v13 Stage G1a: frozen-selector budget stress
Upload the v13 source archive and the complete F2 artifact. G1a reuses verified F2 captures and the B=18 reference, then evaluates only B=6 and B=10. Every fresh result is mirrored to Drive.

In [ ]:
from pathlib import Path
import json, os, shutil, subprocess, sys, zipfile
import torch
IS_COLAB = 'google.colab' in sys.modules
if not torch.cuda.is_available():
    raise RuntimeError('Select a GPU runtime before continuing.')
gpu = torch.cuda.get_device_properties(0)
print('GPU:', gpu.name, '| VRAM GiB:', round(gpu.total_memory / 2**30, 1))

In [ ]:
if IS_COLAB:
    from google.colab import drive, files
    drive.mount('/content/drive')
    files.upload()
    uploaded_paths = list(Path('/content').glob('*.zip'))
    WORK = Path('/content/bikv_v13_g1_work')
    MIRROR = Path('/content/drive/MyDrive/bikv_runs')
else:
    WORK = Path.cwd() / 'bikv_v13_g1_work'
    MIRROR = None
    uploaded_paths = list(Path.cwd().glob('*.zip'))
source_archives = [p for p in uploaded_paths if 'v13_stage_g1' in p.name and p.suffix == '.zip']
f2_archives = [p for p in uploaded_paths if 'v12-stage-f2-medium' in p.name and p.suffix == '.zip']
if not source_archives or not f2_archives:
    print('ZIP files found:', [p.name for p in uploaded_paths])
    raise FileNotFoundError('Upload boundary_interaction_kv_v13_stage_g1.zip and bikv-v12-stage-f2-medium-seed-20261014_artifacts.zip')
source_archive = max(source_archives, key=lambda p: p.stat().st_mtime)
f2_archive = max(f2_archives, key=lambda p: p.stat().st_mtime)
print('Using source archive:', source_archive.name)
print('Using F2 artifact:', f2_archive.name)
SOURCE = WORK / 'source_v13_stage_g1'
F2 = WORK / 'f2_artifacts'
for destination, archive_path in ((SOURCE, source_archive), (F2, f2_archive)):
    if destination.exists():
        shutil.rmtree(destination)
    destination.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(archive_path) as archive:
        archive.extractall(destination)
projects = list(SOURCE.rglob('pyproject.toml'))
statuses = list(F2.rglob('status.json'))
if len(projects) != 1 or len(statuses) != 1:
    raise RuntimeError(f'Expected one project and one F2 run, found {len(projects)} and {len(statuses)}')
PROJECT = projects[0].parent
F2_RUN = statuses[0].parent
print('Project:', PROJECT)
print('F2 source:', F2_RUN)

In [ ]:
os.environ['HF_HOME'] = '/content/hf_cache'
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(PROJECT)], check=True)
if sys.version_info >= (3, 13):
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', 'fire==0.7.1'], check=True)
print('Installed project and pinned dependencies.')

In [ ]:
base = json.loads((PROJECT / 'configs' / 'v13_stage_g1_budget.json').read_text())
base['data_path'] = str(WORK / 'data' / 'stage_f2_medium_seed_20261014.jsonl')
base['frozen_models_path'] = str(PROJECT / 'configs' / 'v12_stage_f2_frozen_models.json')
base['source_run_root'] = str(F2_RUN)
base['output_root'] = str(WORK / 'runs')
base['mirror_root'] = str(MIRROR) if MIRROR else None
runtime_path = WORK / 'runtime_v13_stage_g1.json'
runtime_path.parent.mkdir(parents=True, exist_ok=True)
runtime_path.write_text(json.dumps(base, indent=2), encoding='utf-8')
def stream(command):
    print(' '.join(map(str, command)), flush=True)
    env = {**os.environ, 'PYTHONUNBUFFERED': '1', 'PYTHONPATH': f"{PROJECT / 'src'}:{PROJECT / 'scripts'}"}
    process = subprocess.Popen(command, cwd=PROJECT, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, env=env)
    for line in process.stdout:
        print(line, end='', flush=True)
    code = process.wait()
    if code:
        raise subprocess.CalledProcessError(code, command)
stream([sys.executable, '-u', '-m', 'bikv.cli', 'prepare', '--config', str(runtime_path)])
print(runtime_path.read_text())

## G1a GPU run
The runner validates every F2 capture and result before loading the model. Re-run this cell after a disconnect; checkpoints are restored from Drive and completed evaluations are skipped.

In [ ]:
stream([sys.executable, '-u', str(PROJECT / 'scripts' / 'stage_g1_budget.py'), '--config', str(runtime_path)])

In [ ]:
run_root = WORK / 'runs' / base['run_id']
status = run_root / 'status.json'
print(status.read_text() if status.exists() else 'status not written yet')
if status.exists() and json.loads(status.read_text()).get('state') == 'COMPLETE':
    stream([sys.executable, '-u', str(PROJECT / 'scripts' / 'analyze_stage_g1.py'), str(run_root)])
    print((run_root / 'analysis' / 'report.md').read_text())
    from bikv.storage import ResultStore
    ResultStore(base['output_root'], base['run_id'], base['mirror_root']).bundle()
bundle = WORK / 'runs' / f"{base['run_id']}_artifacts.zip"
print('Bundle:', bundle, '| exists:', bundle.exists())
if IS_COLAB and bundle.exists():
    files.download(str(bundle))